# Pay for Data — Heurist Finance Agent

## 개요

**Amazon Bedrock AgentCore payments**를 사용하여 실시간 market data에 결제하는 finance research agent입니다. Agent는 실시간 가격, SEC filing, macro indicator를 제공하는 유료 [Heurist](https://heurist.xyz) endpoint를 호출하고 AgentCore Code Interpreter로 data를 분석한 다음 chart와 report를 S3 presigned URL로 반환합니다. Tool에는 수동 payment 코드가 전혀 필요하지 않습니다.

Agent는 HTTPS invocation, SigV4 auth, CloudWatch를 통한 자동 observability를 제공하는 managed container endpoint인 **AgentCore Runtime**에 배포됩니다.

### 사용 사례 세부 정보

| 항목 | 세부 정보 |
|:---|:---|
| 사용 사례 유형 | 자동 micropayment를 사용하는 agentic data retrieval |
| Agent 유형 | Single |
| Payment protocol | x402(HTTP 402 Payment Required) |
| Agentic framework | [Strands Agents](https://strandsagents.com/) |
| LLM model | Amazon Bedrock의 Claude Sonnet 4.6(구성 가능) |
| 사용 SDK | `bedrock-agentcore[strands-agents]`(public PyPI) |
| Wallet 유형 | Embedded crypto wallet(Coinbase CDP) |
| Payment network | Base Sepolia testnet(USDC) — `.env`의 `NETWORK`를 통해 mainnet으로 전환 |

### 아키텍처

```
Notebook (ManagementRole)                AgentCore Runtime (ProcessPaymentRole)
  |                                        +------------------------------+
  | create_payment_session(budget=$X)      |  runtime_agent.py            |
  |                                        |  BedrockAgentCoreApp         |
  |-- invoke_agent_runtime(           -->  |  + AgentCorePaymentsPlugin   |
  |     manager_arn, session_id,           |                              |
  |     instrument_id, prompt)             |  http_request -> 402         |
  |                                        |  -> ProcessPayment -> retry  |
  |<-- {response, artifacts: [{url}]} ---  |  -> Code Interpreter         |
  |                                        |  -> export to S3             |
  | get_payment_session(check spend)       +------------------------------+
                                                      |
                                                      v
                                          CloudWatch GenAI Observability
                                          (automatic via OpenTelemetry)
```

### 확인할 AgentCore 기능

| 기능 | 사용 방식 |
|:---|:---|
| **Payment manager** | 모든 payment activity를 authorize하고 추적하는 중앙 resource |
| **Payment instrument** | Embedded crypto wallet(Coinbase CDP, Base의 USDC) |
| **Payment session** | 시간과 budget이 제한된 authorization(`maxSpendAmount`) |
| **Payment processing** | End-to-end x402 negotiation, proof 생성, retry, on-chain settlement |
| **AgentCore Runtime** | HTTPS endpoint와 SigV4 auth를 제공하는 managed container hosting |
| **AgentCore Code Interpreter** | pandas/matplotlib 분석을 위한 remote sandboxed Python environment |
| **Observability** | CloudWatch GenAI dashboard의 자동 OTel trace + log |

### Notebook 진행 순서

| 단계 | 수행 작업 |
|------|-------------|
| 1 | Credentials 구성 및 AWS identity 확인 |
| 2 | Heurist tool catalog 동기화 |
| 3 | S3 artifact bucket 생성 |
| 4 | Embedded wallet resource provision(CredentialProvider → Manager → Connector → Instrument) |
| 5 | Wallet 입금 및 WalletHub delegation 완료 |
| 6 | Payment Manager observability 활성화(vended log delivery) |
| 7 | AgentCore Runtime에 배포 |
| 8 | Execution role 권한 부여 |
| 9 | 배포된 agent 호출 |
| 10 | CloudWatch에서 observability 확인 |
| 11 | 리소스 정리 |

**실행 전:**
1. `pip install -r requirements.txt`
2. `cp .env.example .env`를 실행하고 Coinbase CDP credentials 및 IAM role ARN 입력
3. Node.js 20+, Docker, AWS CDK가 설치되어 있는지 확인
4. CDP project에서 **Delegated Signing** 활성화: [portal.cdp.coinbase.com](https://portal.cdp.coinbase.com) → project → **Wallet** → **Embedded Wallets** → **Policies** → **Delegated signing** 활성화

전체 설정 세부 정보는 [`README.md`](README.md)를 참조하세요.

## Dependency 설치

In [ ]:
%pip install -r requirements.txt --quiet

## 1단계 — Credentials 구성

Credentials를 `.env`에서 불러오고 AWS identity를 확인합니다. `.env.example`을 `.env`로 복사하고 값을 입력하세요.

이 단계에서는 region, account, role ARN만 검증하며 payment resource ID는 4단계에서 채웁니다.

In [ ]:
import os
import json
import time
import uuid
from datetime import datetime
from dotenv import load_dotenv
import boto3
from boto3.session import Session

load_dotenv(override=True)

REGION = os.environ.get("AWS_REGION", "us-west-2")

# ── endpoint 설정 ──────────────────────────────────────────────────────────
CP_ENDPOINT = os.environ.get("CP_ENDPOINT", f"https://bedrock-agentcore-control.{REGION}.amazonaws.com")
DP_ENDPOINT = os.environ.get("DP_ENDPOINT", f"https://bedrock-agentcore.{REGION}.amazonaws.com")

# ── Coinbase CDP credentials(embedded wallet에 필요) ─────────────────────
CDP_API_KEY_NAME = os.environ["CDP_API_KEY_NAME"]
CDP_API_KEY_PRIVATE_KEY = os.environ["CDP_API_KEY_PRIVATE_KEY"]
CDP_WALLET_SECRET = os.environ["CDP_WALLET_SECRET"]
WALLET_EMAIL = os.environ.get("WALLET_EMAIL", "")

# ── IAM role 설정 ──────────────────────────────────────────────────────────
MANAGEMENT_ROLE_ARN = os.environ["MANAGEMENT_ROLE_ARN"]
PROCESS_PAYMENT_ROLE_ARN = os.environ["PROCESS_PAYMENT_ROLE_ARN"]
CONTROL_PLANE_ROLE_ARN = os.environ["CONTROL_PLANE_ROLE_ARN"]
RESOURCE_RETRIEVAL_ROLE_ARN = os.environ["RESOURCE_RETRIEVAL_ROLE_ARN"]

# ── 이전에 provision된 resource ID(재실행 시 .env에서 load) ──────────────
MANAGER_ARN = os.environ.get("MANAGER_ARN", "")
PAYMENT_CONNECTOR_ID = os.environ.get("PAYMENT_CONNECTOR_ID", "")
PAYMENT_INSTRUMENT_ID = os.environ.get("PAYMENT_INSTRUMENT_ID", "")

# ── Session 설정 ───────────────────────────────────────────────────────────
USER_ID = os.environ.get("USER_ID", "heurist-demo-user")
SESSION_MAX_SPEND = os.environ.get("SESSION_MAX_SPEND", "0.25")
SESSION_EXPIRY_MINUTES = int(os.environ.get("SESSION_EXPIRY_MINUTES", "60"))

# ── network / blockchain 설정 ─────────────────────────────────────────────
# base-mainnet: eip155:8453   (기본값 — mainnet, 실제 USDC on-chain settlement)
# base-sepolia: eip155:84532  (testnet — 무료 faucet, 현재 Heurist Sepolia x402의 EIP-712 simulation 실패)
NETWORK_ALIAS = os.environ.get("NETWORK", "base-mainnet")
NETWORK_MAP = {
    "base-sepolia": {
        "caip2": "eip155:84532",
        "botocore_net": "ETHEREUM",
        "usdc_address": "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
        "chain_enum": "BASE_SEPOLIA",
    },
    "base-mainnet": {
        "caip2": "eip155:8453",
        "botocore_net": "ETHEREUM",
        "usdc_address": "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913",
        "chain_enum": "BASE_MAINNET",
    },
}
if NETWORK_ALIAS not in NETWORK_MAP:
    raise ValueError(f"Unknown NETWORK '{NETWORK_ALIAS}'. Valid: {list(NETWORK_MAP)}")
ACTIVE_NETWORK = NETWORK_MAP[NETWORK_ALIAS]

# ── Bedrock model 설정 ────────────────────────────────────────────────────
BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-6")

# ── AWS client 설정 ───────────────────────────────────────────────────────
base_session = Session(region_name=REGION)
sts = base_session.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]


def assume_role(role_arn: str, session_name: str) -> Session:
    creds = sts.assume_role(RoleArn=role_arn, RoleSessionName=session_name)["Credentials"]
    sess = Session(
        aws_access_key_id=creds["AccessKeyId"],
        aws_secret_access_key=creds["SecretAccessKey"],
        aws_session_token=creds["SessionToken"],
        region_name=REGION,
    )
    assumed_arn = sess.client("sts").get_caller_identity()["Arn"]
    print(f"  → {assumed_arn}")
    return sess


# 컨트롤 플레인 클라이언트: CreatePaymentCredentialProvider / Manager / Connector
print("Assuming ControlPlaneRole...")
cp_session = assume_role(CONTROL_PLANE_ROLE_ARN, f"cp-setup-{int(datetime.now().timestamp())}")
cp_client = cp_session.client("bedrock-agentcore-control", endpoint_url=CP_ENDPOINT)
print("✅ CP client ready")

# 관리 클라이언트: CreatePaymentSession / GetPaymentSession / InvokeAgentRuntime
print("Assuming ManagementRole...")
mgmt_session = assume_role(MANAGEMENT_ROLE_ARN, f"heurist-mgmt-{int(datetime.now().timestamp())}")
mgmt_client = mgmt_session.client("bedrock-agentcore", endpoint_url=DP_ENDPOINT)
print("✅ Management client ready")

print(f"\nRegion:   {REGION}")
print(f"Account:  {ACCOUNT_ID}")
print(f"Network:  {NETWORK_ALIAS} ({ACTIVE_NETWORK['caip2']})")
print(f"Model:    {BEDROCK_MODEL_ID}")
if MANAGER_ARN:
    print(f"Manager ARN loaded from .env — Step 4 will be skipped: {MANAGER_ARN}")

## 2단계 — Heurist Tool Catalog 동기화

Heurist mesh에서 x402 지원 endpoint의 현재 registry를 가져와 로컬에 cache합니다. Runtime container image는 build 시 이 cache를 포함하므로 배포된 agent가 시작할 때 Heurist registry를 호출하지 않고 catalog를 읽습니다.

Container에 최신 catalog가 포함되도록 배포 전에 이 셀을 다시 실행하세요.

In [ ]:
import sys

sys.path.insert(0, "agent")
from catalog import fetch_live_catalog, get_tools_for_agents
from config import DEFAULT_HEURIST_AGENT_IDS

HEURIST_AGENT_IDS = (
    tuple(a.strip() for a in os.environ.get("HEURIST_AGENT_IDS", "").split(",") if a.strip())
    or DEFAULT_HEURIST_AGENT_IDS
)

catalog = fetch_live_catalog()
selected = get_tools_for_agents(HEURIST_AGENT_IDS)

print(f"Agents in registry: {catalog['count']}")
print(f"Selected agents:    {', '.join(HEURIST_AGENT_IDS)}")
print(f"Loaded paid tools:  {len(selected)}")
print()
for t in selected:
    print(f"  {t['agent_id']:30s}  {t['tool_name']:35s}  ${t['price_usd']:.3f}")

## 3단계 — S3 Artifact Bucket 생성

Agent가 Code Interpreter에서 생성한 chart, report, CSV는 S3에 upload됩니다. Agent는 `CI_ARTIFACTS_TTL`초 동안 유효한 presigned download URL을 반환합니다(기본값: 1시간).

Bucket은 private이며 public access가 허용되지 않습니다. 이미 bucket이 있다면 이 셀을 건너뛰고 `ARTIFACTS_BUCKET`을 해당 이름으로 설정하세요.

In [ ]:
ARTIFACTS_BUCKET = os.environ.get(
    "ARTIFACTS_BUCKET",
    f"heurist-finance-artifacts-{ACCOUNT_ID}-{REGION}",
)

s3 = boto3.client("s3", region_name=REGION)
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=ARTIFACTS_BUCKET)
    else:
        s3.create_bucket(
            Bucket=ARTIFACTS_BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    s3.put_public_access_block(
        Bucket=ARTIFACTS_BUCKET,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True,
            "IgnorePublicAcls": True,
            "BlockPublicPolicy": True,
            "RestrictPublicBuckets": True,
        },
    )
    print(f"Created bucket: {ARTIFACTS_BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists: {ARTIFACTS_BUCKET}")

print(f"Artifacts will be stored at: s3://{ARTIFACTS_BUCKET}/heurist-finance-artifacts/")

## 4단계 — Embedded Wallet Resource Provision

이 셀은 **사용자별로 한 번** 실행하여 AgentCore payments resource stack을 생성합니다.
**CredentialProvider → PaymentManager → PaymentConnector → EmbeddedCryptoWallet Instrument**

이전 실행에서 얻은 `MANAGER_ARN`, `PAYMENT_CONNECTOR_ID`, `PAYMENT_INSTRUMENT_ID`가 있다면 `.env`에 설정하고 5단계로 이동합니다.

> **사전 요구 사항:** 이 셀을 실행하기 전에 CDP project에서 **Delegated Signing**을 활성화하세요.
> [portal.cdp.coinbase.com](https://portal.cdp.coinbase.com) → project → **Wallet** → **Embedded Wallets** → **Policies**로 이동하여 **Delegated signing**을 활성화합니다.
> 이 설정이 없으면 `ProcessPayment`가 delegated signing error와 함께 실패합니다.

In [ ]:
# ── 4a. 자격 증명 공급자 ──────────────────────────────────────────────────
# Coinbase CDP API key를 AgentCore에 안전하게 저장
# StripePrivy의 경우 credentialProviderVendor="StripePrivy"로 설정하고
# coinbaseCdpConfiguration을 stripePlatformConfiguration으로 교체
if MANAGER_ARN:
    print(f"MANAGER_ARN already set — skipping credential provider creation.")
    print(f"  Manager ARN: {MANAGER_ARN}")
    CREDENTIAL_PROVIDER_ARN = "(loaded from .env — not re-created)"
else:
    cred_resp = cp_client.create_payment_credential_provider(
        name=f"HeuristCdp{int(time.time())}",
        credentialProviderVendor="CoinbaseCDP",
        providerConfigurationInput={
            "coinbaseCdpConfiguration": {
                "apiKeyId": CDP_API_KEY_NAME,
                "apiKeySecret": CDP_API_KEY_PRIVATE_KEY,
                "walletSecret": CDP_WALLET_SECRET,
            }
        },
    )
    CREDENTIAL_PROVIDER_ARN = cred_resp["credentialProviderArn"]
    print(f"✅ Credential Provider: {CREDENTIAL_PROVIDER_ARN}")

In [ ]:
# ── 4b. 결제 관리자 ───────────────────────────────────────────────────────
if MANAGER_ARN:
    print(f"Reusing Manager ARN from .env: {MANAGER_ARN}")
    MANAGER_ID = MANAGER_ARN.split("/")[-1]
else:
    mgr_resp = cp_client.create_payment_manager(
        name=f"HeuristPayMgr{int(time.time())}",
        description="AgentCore payments - Heurist Finance Agent",
        authorizerType="AWS_IAM",
        roleArn=RESOURCE_RETRIEVAL_ROLE_ARN,
        clientToken=str(uuid.uuid4()),
    )
    MANAGER_ARN = mgr_resp["paymentManagerArn"]
    MANAGER_ID = mgr_resp["paymentManagerId"]
    print(f"✅ Payment Manager ARN: {MANAGER_ARN}")
    print(f"   Manager ID:          {MANAGER_ID}")
    print("\n📋 Save to .env:")
    print(f"   MANAGER_ARN={MANAGER_ARN}")

In [ ]:
# ── 4c. 결제 커넥터 ───────────────────────────────────────────────────────
if PAYMENT_CONNECTOR_ID:
    print(f"Reusing Payment Connector from .env: {PAYMENT_CONNECTOR_ID}")
else:
    conn_resp = cp_client.create_payment_connector(
        paymentManagerId=MANAGER_ID,
        name=f"HeuristCoinbaseConn{int(time.time())}",
        description="Coinbase CDP connector for Heurist Finance Agent",
        type="CoinbaseCDP",
        credentialProviderConfigurations=[{"coinbaseCDP": {"credentialProviderArn": CREDENTIAL_PROVIDER_ARN}}],
        clientToken=str(uuid.uuid4()),
    )
    PAYMENT_CONNECTOR_ID = conn_resp["paymentConnectorId"]
    print(f"✅ Payment Connector ID: {PAYMENT_CONNECTOR_ID}")
    print("\n📋 Save to .env:")
    print(f"   PAYMENT_CONNECTOR_ID={PAYMENT_CONNECTOR_ID}")

In [ ]:
# ── 4d. 내장 암호화폐 지갑 결제 수단 ─────────────────────────────────────
# AgentCore가 on-chain wallet을 provision하므로 기존 CDP wallet 불필요
# WALLET_EMAIL이 linkedAccounts를 통해 wallet을 사용자 identity에 연결
if PAYMENT_INSTRUMENT_ID:
    print(f"Reusing Payment Instrument from .env: {PAYMENT_INSTRUMENT_ID}")
    WALLET_HUB_URL = ""
else:
    linked_accounts = []
    if WALLET_EMAIL:
        linked_accounts = [{"email": {"emailAddress": WALLET_EMAIL}}]

    inst_resp = mgmt_client.create_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
        userId=USER_ID,
        paymentInstrumentType="EMBEDDED_CRYPTO_WALLET",
        paymentInstrumentDetails={
            "embeddedCryptoWallet": {
                "network": ACTIVE_NETWORK["botocore_net"],
                "linkedAccounts": linked_accounts,
            }
        },
        clientToken=str(uuid.uuid4()),
    )
    instrument = inst_resp["paymentInstrument"]
    PAYMENT_INSTRUMENT_ID = instrument["paymentInstrumentId"]
    wallet_details = instrument.get("paymentInstrumentDetails", {}).get("embeddedCryptoWallet", {})
    WALLET_ADDRESS = wallet_details.get("walletAddress", "<pending>")
    WALLET_HUB_URL = wallet_details.get("redirectUrl", "")

    print(f"✅ Payment Instrument ID: {PAYMENT_INSTRUMENT_ID}")
    print(f"   Wallet Address:        {WALLET_ADDRESS}")
    print(f"   Network:               {ACTIVE_NETWORK['caip2']}")
    if WALLET_HUB_URL:
        print(f"   WalletHub URL:         {WALLET_HUB_URL}")
    print("\n📋 Save to .env:")
    print(f"   PAYMENT_INSTRUMENT_ID={PAYMENT_INSTRUMENT_ID}")

## 5단계 — Wallet 입금 및 Signing Delegation 부여

AgentCore에서 embedded wallet을 provision했습니다. Agent가 payment를 수행하기 전에 **두 가지** 설정 작업을 완료합니다.

### 5a — Signing 권한 부여(WalletHub)

1. 위에 출력된 **WalletHub URL**을 browser에서 엽니다.
2. 구성한 `WALLET_EMAIL`로 로그인합니다.
3. Wallet을 확인하고 **Grant signing permission**을 선택합니다.

이 단계를 완료하기 전에는 agent가 transaction에 sign할 수 없습니다. 이를 통해 [Coinbase CDP Delegated Signing](https://portal.cdp.coinbase.com)이 활성화됩니다. AgentCore는 private key를 보유하지 않고 `ProcessPayment`를 호출하여 사용자를 대신해 transaction proof를 생성합니다.

> **WalletHub URL이 반환되지 않았고 instrument status가 `ACTIVE`이면** wallet에 권한이 이미 부여된 것입니다. 바로 입금을 진행하세요.

### 5b — Wallet 입금

Wallet address로 testnet USDC를 전송합니다.
- **Base Sepolia:** go to https://faucet.circle.com → select *Base Sepolia* → paste the wallet address

Mainnet(`NETWORK=base-mainnet`)에서는 onramp를 통해 입금하거나 다른 wallet에서 USDC를 전송합니다.

### 5c — Balance 검증

입금 후(faucet transaction은 약 30초 소요) 아래 셀을 실행하여 balance를 확인합니다. 0이 표시되면 transaction이 아직 pending 상태일 수 있으므로 다시 실행하세요.

In [ ]:
# GetPaymentInstrumentBalance 호출을 위해 잠시 ProcessPaymentRole 사용
# 배포된 Runtime container는 동일한 role을 자동 사용
print("Assuming ProcessPaymentRole for balance check...")
pp_session = assume_role(
    PROCESS_PAYMENT_ROLE_ARN,
    f"heurist-balance-check-{int(datetime.now().timestamp())}",
)
pp_client = pp_session.client("bedrock-agentcore", endpoint_url=DP_ENDPOINT)

try:
    balance_resp = pp_client.get_payment_instrument_balance(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
        paymentInstrumentId=PAYMENT_INSTRUMENT_ID,
        userId=USER_ID,
        chain=ACTIVE_NETWORK["chain_enum"],
        token="USDC",
    )
    token_balance = balance_resp.get("tokenBalance", {})
    if token_balance:
        amount_units = int(token_balance.get("amount", 0))
        decimals = token_balance.get("decimals", 6)
        readable = amount_units / (10**decimals)
        print(
            f"✅ Wallet balance: {readable:.6f} {token_balance.get('token', 'USDC')} on {token_balance.get('chain', ACTIVE_NETWORK['chain_enum'])}"
        )
        if readable == 0:
            print("   ⚠️  Balance is 0 — faucet transaction may still be pending. Wait ~30s and re-run.")
    else:
        print("⚠️  Balance returned empty — faucet may still be pending.")
    print(f"   Instrument ID: {PAYMENT_INSTRUMENT_ID}")
except Exception as e:
    print(f"⚠️  GetPaymentInstrumentBalance failed: {e}")
    print("   Ensure bedrock-agentcore:GetPaymentInstrumentBalance is in the ProcessPaymentRole policy.")
    print("   Continue to Step 6 if the wallet is funded.")

## 6단계 — Payment Manager Observability 활성화

Payment Manager telemetry는 vended log delivery를 통한 **opt-in** 방식입니다. 활성화하면 Payment Manager가 session, transaction, API별 metric, *Agents using Payments* attribution counter와 함께 **AgentCore Observability → Payments** dashboard에 표시됩니다.

이 셀은 **idempotent**하게 동작하여 이미 존재하는 resource는 건너뜁니다. Manager를 생성한 후 한 번 실행하세요.

In [ ]:
logs = boto3.client("logs", region_name=REGION)

# Manager의 short ID(첫 hyphen group 앞의 첫 segment)를
# DNS-safe log group component로 사용
mgr_short_id = MANAGER_ARN.split("/")[-1].split("-")[0]
LG_NAME = f"/aws/vendedlogs/bedrock-agentcore/heurist-{mgr_short_id}"
LG_ARN = f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LG_NAME}"

SRC_LOGS_NAME = f"heurist-payments-logs-{mgr_short_id}"
SRC_TRACES_NAME = f"heurist-payments-traces-{mgr_short_id}"
DST_LOGS_NAME = f"heurist-payments-logs-dest-{mgr_short_id}"
DST_TRACES_NAME = f"heurist-payments-traces-dest-{mgr_short_id}"

# 0단계 — log group
try:
    logs.create_log_group(logGroupName=LG_NAME)
    logs.put_retention_policy(logGroupName=LG_NAME, retentionInDays=30)
    print(f"✅ Created log group {LG_NAME}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"   ↻ Log group exists: {LG_NAME}")

# 1단계 — application log delivery source
try:
    logs.put_delivery_source(
        name=SRC_LOGS_NAME,
        resourceArn=MANAGER_ARN,
        logType="APPLICATION_LOGS",
    )
    print(f"✅ Logs delivery source created ({SRC_LOGS_NAME})")
except logs.exceptions.ConflictException:
    print(f"   ↻ Logs delivery source exists ({SRC_LOGS_NAME})")

# 2단계 — trace delivery source
try:
    logs.put_delivery_source(
        name=SRC_TRACES_NAME,
        resourceArn=MANAGER_ARN,
        logType="TRACES",
    )
    print(f"✅ Traces delivery source created ({SRC_TRACES_NAME})")
except logs.exceptions.ConflictException:
    print(f"   ↻ Traces delivery source exists ({SRC_TRACES_NAME})")

# 3a단계 — CloudWatch Logs delivery destination
try:
    logs.put_delivery_destination(
        name=DST_LOGS_NAME,
        deliveryDestinationType="CWL",
        deliveryDestinationConfiguration={"destinationResourceArn": LG_ARN},
    )
    print(f"✅ Logs delivery destination created ({DST_LOGS_NAME})")
except logs.exceptions.ConflictException:
    print(f"   ↻ Logs delivery destination exists ({DST_LOGS_NAME})")

# 3b단계 — X-Ray trace destination
try:
    logs.put_delivery_destination(
        name=DST_TRACES_NAME,
        deliveryDestinationType="XRAY",
    )
    print(f"✅ Traces delivery destination created ({DST_TRACES_NAME})")
except logs.exceptions.ConflictException:
    print(f"   ↻ Traces delivery destination exists ({DST_TRACES_NAME})")

# 4a단계 — log source → destination 연결
try:
    logs.create_delivery(
        deliverySourceName=SRC_LOGS_NAME,
        deliveryDestinationArn=(f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:{DST_LOGS_NAME}"),
    )
    print("✅ Logs delivery created")
except logs.exceptions.ConflictException:
    print("   ↻ Logs delivery exists")

# 4b단계 — trace source → destination 연결
try:
    logs.create_delivery(
        deliverySourceName=SRC_TRACES_NAME,
        deliveryDestinationArn=(f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:{DST_TRACES_NAME}"),
    )
    print("✅ Traces delivery created")
except logs.exceptions.ConflictException:
    print("   ↻ Traces delivery exists")

print()
print("Payment Manager observability enabled.")
print("After your first invocation, the AgentCore Observability → Payments tab will populate.")

## 7단계 — AgentCore Runtime에 배포

`@aws/agentcore` CLI는 project를 scaffold하고 Docker image를 build하여 ECR에 push한 다음 CDK를 통해 배포합니다. 첫 배포에는 약 5~10분이 걸립니다.

> **사전 요구 사항:** Node.js 20+, 실행 중인 Docker, 설치된 AWS CDK
>
> **비용 안내:** 유료 AWS resource(ECR, Runtime endpoint, CloudWatch log)가 생성됩니다. 작업을 마치면 리소스 정리 섹션을 실행하세요.

In [ ]:
import subprocess
import shutil

# Agent source of truth는 `agent/`에 있음. CLI는 AgentCore service가 요구하는
# layout(agentcore.json, aws-targets.json, cdk/)을 가진 별도 project directory인
# `payfordata/`를 scaffold함. 배포할 때마다 `agent/`의 내용을
# `payfordata/app/HeuristFinanceAgent/`로 복사
PROJECT_NAME = "payfordata"  # CLI project 이름 - 소문자 유지
AGENT_NAME = "HeuristFinanceAgent"  # 배포된 agent 이름


def run(cmd, **kw):
    result = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print("stdout:", result.stdout[-1000:])
        print("stderr:", result.stderr[-1000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result


# ── 7a. CLI project scaffold 생성(멱등) ──────────────────────────────────
if not os.path.isdir(PROJECT_NAME):
    print(f"Scaffolding {PROJECT_NAME}/ ...")
    run(
        [
            "agentcore",
            "create",
            "--name",
            AGENT_NAME,
            "--project-name",
            PROJECT_NAME,
            "--defaults",
            "--no-agent",
            "--skip-git",
            "--skip-python-setup",
            "--skip-install",
            "--json",
        ]
    )
    run(
        [
            "agentcore",
            "add",
            "agent",
            "--type",
            "byo",
            "--name",
            AGENT_NAME,
            "--build",
            "Container",
            "--language",
            "Python",
            "--framework",
            "Strands",
            "--model-provider",
            "Bedrock",
            "--code-location",
            f"app/{AGENT_NAME}",
            "--entrypoint",
            "main.py",
            "--network-mode",
            "PUBLIC",
            "--protocol",
            "HTTP",
            "--idle-timeout",
            "600",
            "--max-lifetime",
            "1800",
            "--json",
        ],
        cwd=PROJECT_NAME,
    )
    print(f"✅ Scaffolded {PROJECT_NAME}/")
else:
    print(f"{PROJECT_NAME}/ already exists — skipping create")

# ── 7b. Build context에 `agent/` stage ────────────────────────────────────
# `agent/` 내부의 모든 항목이 container image에 포함됨. Agent 코드 변경을
# 반영하도록 배포할 때마다 복사
build_ctx = f"{PROJECT_NAME}/app/{AGENT_NAME}"
os.makedirs(build_ctx, exist_ok=True)

# Image에 최신 tool URL이 포함되도록 복사 전에 catalog cache 갱신
print("Refreshing Heurist catalog cache...")
run(["python3", "agent/sync_registry.py"])

for fname in (
    "main.py",
    "catalog.py",
    "config.py",
    "sync_registry.py",
    "catalog_live_cache.json",
    "requirements.txt",
    "Dockerfile",
):
    src = os.path.join("agent", fname)
    dst = os.path.join(build_ctx, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
print(f"✅ Staged agent/* → {build_ctx}/")

# ── 7c. Container .env 작성(service config만 포함, payment credentials 제외) ─
runtime_env = f"""# Container image에 포함되는 Runtime 구성
# 결제 자격 증명은 여기에 없으며 invocation payload에서 전달됨
CI_ARTIFACTS_BUCKET={ARTIFACTS_BUCKET}
CI_ARTIFACTS_PREFIX=heurist-finance-artifacts
CI_ARTIFACTS_TTL=3600
AWS_REGION={REGION}
BEDROCK_MODEL_ID={BEDROCK_MODEL_ID}
AGENT_NAME={AGENT_NAME}
HEURIST_AGENT_IDS={",".join(HEURIST_AGENT_IDS)}

# strands_tools.http_request의 POST/PUT/DELETE 대화형 확인 prompt를 건너뜀
# Runtime container에는 TTY가 없으므로 필요함
BYPASS_TOOL_CONSENT=true

# Agent turn당 Bedrock 최대 output token 수. 60_000이면 multi-step workflow
# (데이터 조회 + Code Interpreter + chart export + Markdown report)를 완료할
# 여유가 있음. SDK 기본값 4k를 사용하면 실행 중 MaxTokensReachedException 발생
# 짧은 Q&A만 필요하면 낮출 것. Bedrock은 output token당 과금하므로 60k 상한은
# Claude Sonnet 4.6 기준 turn당 최악의 경우 약 $0.90
# (output token 1,000개당 US$0.015). 일반적인 agent turn은 훨씬 적게 사용함
AGENT_MAX_TOKENS=32000
"""
with open(f"{build_ctx}/.env", "w") as f:
    f.write(runtime_env)
print(f"✅ Wrote {build_ctx}/.env")

# ── 7d. agentcore.json에서 executionRoleArn + runtime version 고정 ────────
config_path = os.path.join(PROJECT_NAME, "agentcore", "agentcore.json")
with open(config_path) as f:
    project_config = json.load(f)
for runtime in project_config.get("runtimes", []):
    if runtime.get("name") == AGENT_NAME:
        runtime["executionRoleArn"] = PROCESS_PAYMENT_ROLE_ARN
        runtime["runtimeVersion"] = "PYTHON_3_13"
        break
with open(config_path, "w") as f:
    json.dump(project_config, f, indent=2)
print(f"✅ executionRoleArn = {PROCESS_PAYMENT_ROLE_ARN}")
print("✅ runtimeVersion   = PYTHON_3_13")

# ── 7e. Deployment target 설정(account + region) ──────────────────────────
targets_path = os.path.join(PROJECT_NAME, "agentcore", "aws-targets.json")
with open(targets_path, "w") as f:
    json.dump(
        [
            {
                "name": "default",
                "description": "Heurist Finance Agent — Runtime deployment",
                "account": ACCOUNT_ID,
                "region": REGION,
            }
        ],
        f,
        indent=2,
    )
print(f"✅ Deployment target: {ACCOUNT_ID} / {REGION}")

# ── 7f. CDK npm dependency 설치(CLI가 배포 시 사용) ──────────────────────
cdk_dir = os.path.join(PROJECT_NAME, "agentcore", "cdk")
if os.path.isdir(cdk_dir) and not os.path.isdir(os.path.join(cdk_dir, "node_modules")):
    print(f"Installing CDK npm deps in {cdk_dir}/ ...")
    run(["npm", "install", "--silent"], cwd=cdk_dir)
    print("✅ CDK deps installed")

In [ ]:
# 배포(첫 실행 약 5~10분 — CodeBuild가 container image build)
print("Deploying to AgentCore Runtime — this can take 5–10 minutes (CodeBuild)...")
run(["agentcore", "deploy", "--yes"], cwd=PROJECT_NAME)
print("✅ Agent deployed")

In [ ]:
# 배포된 agent runtime ARN 저장
status_proc = subprocess.run(
    ["agentcore", "status", "--type", "agent", "--json"],
    cwd=PROJECT_NAME,
    capture_output=True,
    text=True,
    check=True,
)
status = json.loads(status_proc.stdout)
entries = status if isinstance(status, list) else status.get("resources", [])

AGENT_RUNTIME_ARN = None
for entry in entries:
    name = entry.get("name") or entry.get("agentName")
    if name == AGENT_NAME:
        AGENT_RUNTIME_ARN = entry.get("agentRuntimeArn") or entry.get("runtimeArn") or entry.get("arn")
        break

if not AGENT_RUNTIME_ARN:
    print("Raw status output:")
    print(json.dumps(status, indent=2))
    raise RuntimeError("Could not locate agent runtime ARN in status output")

print(f"✅ Agent Runtime ARN: {AGENT_RUNTIME_ARN}")

## 8단계 — Execution Role 권한 부여

지정한 `PROCESS_PAYMENT_ROLE_ARN`이 container의 execution role이 됩니다. 세 가지 추가 policy를 연결합니다.

1. **Payment data-plane** — `ProcessPayment` 및 read operation(payment manager로 scope 제한)
2. **Code Interpreter** — `StartCodeInterpreterSession`, `InvokeCodeInterpreter`, `StopCodeInterpreterSession`
3. **S3 artifact** — artifact bucket prefix로 scope가 제한된 `PutObject` + `GetObject`

Execution role은 payment session 또는 instrument를 생성할 수 없으며 해당 권한은 `ManagementRole`에 유지됩니다.

In [ ]:
iam = boto3.client("iam")

# ARN에서 role name 추출
RUNTIME_ROLE_NAME = PROCESS_PAYMENT_ROLE_ARN.split("/")[-1]
print(f"Execution role: {RUNTIME_ROLE_NAME}")

# ── 8-0. AgentCore Runtime의 execution role assume 가능 여부 확인 ───────
# Runtime infrastructure가 container 시작 시 이 role을 assume
# Trust policy에 bedrock-agentcore.amazonaws.com이 없으면 CDK stack에서
# 다음 오류가 발생함: 'Role validation failed — trust policy allows assumption
# by this service' (BedrockAgentCoreControl, Status Code: 400).
import json as _json

current_trust = iam.get_role(RoleName=RUNTIME_ROLE_NAME)["Role"]["AssumeRolePolicyDocument"]
principals = [s.get("Principal", {}) for s in current_trust.get("Statement", [])]
service_principals = [p.get("Service", "") for p in principals if isinstance(p, dict)]
service_principal_flat = [s for item in service_principals for s in ([item] if isinstance(item, str) else item)]
if "bedrock-agentcore.amazonaws.com" not in service_principal_flat:
    current_trust["Statement"].append(
        {
            "Sid": "AllowAgentCoreRuntimeService",
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    )
    iam.update_assume_role_policy(
        RoleName=RUNTIME_ROLE_NAME,
        PolicyDocument=_json.dumps(current_trust),
    )
    print("✅ Added bedrock-agentcore.amazonaws.com to execution role trust policy")
else:
    print("   ↻ bedrock-agentcore.amazonaws.com already in trust policy")

# 1. Payment data-plane(bare payment-manager/* ARN에 유의. 이 ARN이 없으면
#    instrument scope의 resource pattern이 manager 자체를 포함하지 않아 plugin의
#    GetPaymentInstrument 호출에서 AccessDeniedException 발생)
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="HeuristPaymentDataPlaneAccess",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "PaymentDataPlaneAccess",
                    "Effect": "Allow",
                    "Action": [
                        "bedrock-agentcore:ProcessPayment",
                        "bedrock-agentcore:GetPaymentInstrument",
                        "bedrock-agentcore:GetPaymentInstrumentBalance",
                        "bedrock-agentcore:GetPaymentSession",
                        "bedrock-agentcore:GetResourcePaymentToken",
                    ],
                    "Resource": [
                        f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:payment-manager/*",
                        f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:payment-manager/*/instrument/*",
                        f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:payment-manager/*/session/*",
                    ],
                }
            ],
        }
    ),
)
print("✅ Payment data-plane permissions added")

# 2. Code Interpreter
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="HeuristCodeInterpreterAccess",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "CodeInterpreterAccess",
                    "Effect": "Allow",
                    "Action": [
                        "bedrock-agentcore:StartCodeInterpreterSession",
                        "bedrock-agentcore:StopCodeInterpreterSession",
                        "bedrock-agentcore:InvokeCodeInterpreter",
                    ],
                    "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:code-interpreter/*",
                }
            ],
        }
    ),
)
print("✅ Code Interpreter permissions added")

# 3. S3 아티팩트
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="HeuristS3ArtifactsAccess",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "S3ArtifactsReadWrite",
                    "Effect": "Allow",
                    "Action": ["s3:PutObject", "s3:GetObject"],
                    "Resource": f"arn:aws:s3:::{ARTIFACTS_BUCKET}/heurist-finance-artifacts/*",
                }
            ],
        }
    ),
)
print(f"✅ S3 artifact permissions added (bucket: {ARTIFACTS_BUCKET})")

# 4. Bedrock — cross-region inference-profile ARN 포함
#    us-west-2의 Claude Sonnet 4.6은 CRIS profile을 사용하므로
#    foundation-model/*만 허용하면 InvokeModelWithResponseStream에서 AccessDeniedException 발생
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="HeuristBedrockInvoke",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "BedrockModelInvocation",
                    "Effect": "Allow",
                    "Action": [
                        "bedrock:InvokeModel",
                        "bedrock:InvokeModelWithResponseStream",
                    ],
                    "Resource": [
                        "arn:aws:bedrock:*::foundation-model/*",
                        f"arn:aws:bedrock:*:{ACCOUNT_ID}:inference-profile/*",
                        f"arn:aws:bedrock:*:{ACCOUNT_ID}:application-inference-profile/*",
                    ],
                }
            ],
        }
    ),
)
print("✅ Bedrock model invocation permissions added (incl. inference profiles)")

print()
print("Permissions summary:")
print("  Payment:          ProcessPayment, GetPaymentInstrument, GetPaymentSession")
print("  Code Interpreter: StartSession, StopSession, InvokeCodeInterpreter")
print("  S3:               PutObject + GetObject (artifacts bucket prefix only)")
print("  Bedrock:          InvokeModel(WithResponseStream) on foundation-model + inference-profile/*")
print("  Not granted:      CreateSession, CreateInstrument (ManagementRole only)")

## 9단계 — 배포된 Agent 호출

새 payment session을 생성한 후(app backend가 `ManagementRole`을 통해 budget 제어) 배포된 agent를 호출합니다. Response에는 research summary와 chart 또는 report의 presigned S3 URL이 포함됩니다.

```json
{
  "response": "<markdown research summary>",
  "artifacts": [
    {"name": "chart.png", "url": "https://...", "expires_in": 3600}
  ]
}
```

In [ ]:
# 이 invocation을 위한 새 payment session 생성
session_response = mgmt_client.create_payment_session(
    paymentManagerArn=MANAGER_ARN,
    userId=USER_ID,
    expiryTimeInMinutes=SESSION_EXPIRY_MINUTES,
    limits={
        "maxSpendAmount": {
            "value": SESSION_MAX_SPEND,
            "currency": "USD",
        }
    },
    clientToken=str(uuid.uuid4()),
)

payment_session = session_response["paymentSession"]
SESSION_ID = payment_session["paymentSessionId"]

print("✅ Payment session created")
print(f"   Session ID:     {SESSION_ID}")
print(f"   Budget:         ${SESSION_MAX_SPEND} USD")
print(f"   Expires:        {SESSION_EXPIRY_MINUTES} minutes from now")
if "availableLimits" in payment_session:
    available = payment_session["availableLimits"]["availableSpendAmount"]
    print(f"   Available:      {available['value']} {available['currency']}")

In [ ]:
# InvokeAgentRuntime을 통해 배포된 agent 호출
# Cold start 처리를 위한 retry logic 포함. AgentCore Runtime container는
# idle 후 첫 invoke에서 120초 init timeout을 초과할 수 있음. 첫 시도에서
# container를 warm up하고 이후 시도는 warm 상태에서 실행
from botocore.config import Config as BotoConfig
from botocore.exceptions import ClientError

invoke_payload = {
    "prompt": (
        "Use FredMacroAgent to fetch the latest US GDP growth rate and unemployment rate. "
        "Use Code Interpreter to create a bar chart comparing them and a markdown summary. "
        "Save both as artifacts."
    ),
    "payment_manager_arn": MANAGER_ARN,
    "user_id": USER_ID,
    "payment_session_id": SESSION_ID,
    "payment_instrument_id": PAYMENT_INSTRUMENT_ID,
}

# Cold start 실패가 기본 25분 동안 block되지 않고 빠르게 180초 내에
# 반환되도록 더 짧은 read timeout의 전용 client 사용
# Warm 상태에서는 간단한 prompt의 agent response가 180초 내에 도착
# 복잡한 multi-step workflow(tool 호출 10회 이상 + Code Interpreter)의 경우
# read_timeout을 900으로 늘릴 것
invoke_client = mgmt_session.client(
    "bedrock-agentcore",
    endpoint_url=DP_ENDPOINT,
    config=BotoConfig(read_timeout=900, connect_timeout=10, retries={"max_attempts": 0}),
)

MAX_RETRIES = 3
result = None

for attempt in range(1, MAX_RETRIES + 1):
    print(f"Invoking {AGENT_NAME} (attempt {attempt}/{MAX_RETRIES})...")
    try:
        response = invoke_client.invoke_agent_runtime(
            agentRuntimeArn=AGENT_RUNTIME_ARN,
            payload=json.dumps(invoke_payload).encode("utf-8"),
            contentType="application/json",
            accept="application/json",
        )
        raw = response.get("response", b"")
        result_bytes = raw.read() if hasattr(raw, "read") else raw
        result = json.loads(result_bytes.decode("utf-8")) if result_bytes else {}
        break
    except ClientError as e:
        msg = e.response.get("Error", {}).get("Message", "")
        if "initialization time exceeded" in msg.lower() and attempt < MAX_RETRIES:
            print(f"  ⏳ Container cold-starting — retrying in 15s (attempt {attempt})...")
            time.sleep(15)
        else:
            raise

print("\n── Response ──────────────────────────────────────────────────")
print(result.get("response", result))

artifacts = result.get("artifacts", [])
if artifacts:
    print("\n── Artifacts ─────────────────────────────────────────────────")
    for a in artifacts:
        print(f"  {a['name']}  (expires in {a['expires_in']}s)")
        print(f"  {a['url']}")

In [ ]:
# Session 지출 확인
session_check = mgmt_client.get_payment_session(
    paymentManagerArn=MANAGER_ARN,
    paymentSessionId=SESSION_ID,
    userId=USER_ID,
)

session_data = session_check["paymentSession"]
budget = session_data.get("limits", {}).get("maxSpendAmount", {})
budget_val = float(budget.get("value", 0))
available = session_data.get("availableLimits", {}).get("availableSpendAmount", {})
avail_val = float(available.get("value", budget_val)) if available.get("value") else budget_val
spent = budget_val - avail_val

print("Session spend summary:")
print(f"  Budget:    ${budget_val:.4f} {budget.get('currency', 'USD')}")
print(f"  Remaining: ${avail_val:.4f} {available.get('currency', 'USD')}")
print(f"  Spent:     ${spent:.4f} USD")

## 10단계 — Observability

AgentCore Runtime은 OpenTelemetry를 사용하여 container를 자동 instrument합니다. 첫 invocation 직후 **CloudWatch GenAI Observability**에 trace와 log가 표시됩니다.

![CloudWatch GenAI Observability — Heurist Finance Agent](images/obs-dashboard.png)

각 invocation은 다음 span이 포함된 unified trace를 생성합니다.
- **LLM 호출** — model ID, token 수, latency
- **Tool 호출** — x402 retry 시도를 포함한 `http_request` invocation
- **Agent turn** — 전체 prompt → tool 사용 → response cycle
- **Code Interpreter** — W3C `traceparent` propagation을 통해 `StartCodeInterpreterSession`, `InvokeCodeInterpreter`, `StopCodeInterpreterSession`이 child span으로 표시
- **Payment 호출** — `ProcessPayment`, `GetPaymentInstrument`가 boto3 child span으로 표시

**Payments tab**은 6단계에서 구성한 vended log delivery를 통해 session, transaction, API별 metric, *Agents using Payments* attribution으로 채워집니다(container에 설정한 `AGENT_NAME` env var로 결정).

> **참고:** Payment manager vended log(`ProcessPayment`, `CreateSession` event)는 vended log delivery(6단계)를 통해 별도로 구성합니다. 자세한 내용은 [AgentCore Payments observability 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-observability.html)를 참조하세요.


In [ ]:
print("CloudWatch GenAI Observability dashboard:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core")
print()
print("Stream live logs:")
print(f"  cd {PROJECT_NAME} && agentcore logs")

## 11단계 — 리소스 정리

> ⚠️ 다음 셀은 Runtime deployment와 S3 artifact bucket을 영구적으로 삭제합니다. 계속 진행하기 전에 보관할 artifact를 download하세요.

In [ ]:
# AgentCore Runtime stack 제거(ECR image, CloudWatch log, CDK stack)
run(["agentcore", "remove", "all", "-y"], cwd=PROJECT_NAME)
print("✅ Runtime stack removed")

In [ ]:
# Scaffold된 project directory 제거
if os.path.exists(PROJECT_NAME):
    shutil.rmtree(PROJECT_NAME)
    print(f"Removed {PROJECT_NAME}/")
else:
    print(f"{PROJECT_NAME}/ already removed")

# S3 artifact bucket을 비우고 삭제
s3_resource = boto3.resource("s3")
bucket = s3_resource.Bucket(ARTIFACTS_BUCKET)
try:
    bucket.objects.all().delete()
    bucket.delete()
    print(f"Deleted S3 bucket: {ARTIFACTS_BUCKET}")
except Exception as e:
    print(f"Could not delete bucket: {e}")

### AWS Resource 정리

Payment session은 `expiryTimeInMinutes`가 지나면 자동으로 만료되므로 수동으로 삭제할 필요가 없습니다.

Payment manager, connector, instrument, credential provider를 정리하려면 AWS CLI 또는 boto3를 사용합니다.

```python
# 예제 - 역순으로 삭제
mgmt_client.delete_payment_instrument(paymentManagerArn=MANAGER_ARN, paymentInstrumentId=PAYMENT_INSTRUMENT_ID, userId=USER_ID)
cp_client.delete_payment_connector(paymentManagerId=MANAGER_ID, paymentConnectorId=PAYMENT_CONNECTOR_ID)
cp_client.delete_payment_manager(paymentManagerId=MANAGER_ID)
cp_client.delete_payment_credential_provider(credentialProviderArn=CREDENTIAL_PROVIDER_ARN)
```

---

## 공동 책임

| 책임 항목 | AWS | 사용자 |
|:---|:---:|:---:|
| AgentCore payments service infrastructure 보안 | ✅ | |
| 저장된 payment credentials 암호화 | ✅ | |
| Service level에서 payment session limit 적용 | ✅ | |
| On-chain transaction settlement(Coinbase CDP) | ✅ | |
| Runtime container compute 및 networking 관리 | ✅ | |
| Least-privilege 권한으로 IAM role 구성 | | ✅ |
| 적절한 `maxSpendAmount` payment limit 설정 | | ✅ |
| CDP project에서 Delegated Signing 활성화 | | ✅ |
| AWS credentials 노출 방지 | | ✅ |
| Payment instrument에 충분한 USDC 입금 | | ✅ |
| Agent 지출 및 session 사용량 monitoring | | ✅ |
| Prompt injection 방지를 위한 prompt 검증 | | ✅ |
| Heurist endpoint 이용 약관 검토 | | ✅ |

> **보안 참고:** `.env` 파일 또는 payment credentials를 source control에 commit하지 마세요.
> 공유 deployment에는 AWS Secrets Manager를 사용하세요.
> Payment session은 시간과 budget이 제한됩니다. 자동 또는 unattended context에서 agent를 실행할 때는 보수적인 `maxSpendAmount` limit을 설정하세요.